In [ ]:
# data_explorer.py
import os
import sys
import io
import pandas as pd
import numpy as np
from pathlib import Path

# Dash & Plotly
import dash
from dash import dcc, html, Input, Output, State, ctx
import plotly.express as px

app = dash.Dash(__name__)
server = app.server

# Helper: load dataset
def load_file(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    suffix = p.suffix.lower()
    if suffix in ('.csv', '.txt'):
        return pd.read_csv(p)
    if suffix in ('.xls', '.xlsx'):
        return pd.read_excel(p)
    if suffix in ('.json',):
        return pd.read_json(p)
    raise ValueError("Unsupported file type: " + suffix)

# Default sample dataset
def sample_df():
    rng = np.random.default_rng(42)
    df = pd.DataFrame({
        'x': np.linspace(0, 10, 200),
        'y': np.sin(np.linspace(0, 10, 200)) + rng.normal(scale=0.3, size=200),
        'category': rng.choice(['A','B','C'], size=200),
        'value': rng.integers(0,100,size=200)
    })
    return df

# App layout
app.layout = html.Div([
    html.H2("Data Graph Explorer"),
    html.Div([
        html.Label("Upload file (CSV, Excel, JSON)"),
        dcc.Upload(id='upload-data', children=html.Button("Upload")),
        html.Button("Load sample dataset", id='load-sample'),
        html.Div(id='file-info', style={'marginTop':10})
    ]),
    html.Hr(),
    html.Div(id='controls', children=[
        html.Div([
            html.Label("X column"),
            dcc.Dropdown(id='x-col', placeholder='Select X'),
        ], style={'width':'24%', 'display':'inline-block'}),
        html.Div([
            html.Label("Y column"),
            dcc.Dropdown(id='y-col', placeholder='Select Y'),
        ], style={'width':'24%', 'display':'inline-block', 'marginLeft':'1%'}),
        html.Div([
            html.Label("Color / Group"),
            dcc.Dropdown(id='color-col', placeholder='Optional'),
        ], style={'width':'24%', 'display':'inline-block', 'marginLeft':'1%'}),
        html.Div([
            html.Label("Plot type"),
            dcc.Dropdown(id='plot-type', options=[
                {'label':'Scatter','value':'scatter'},
                {'label':'Line','value':'line'},
                {'label':'Histogram','value':'hist'},
                {'label':'Bar','value':'bar'},
                {'label':'Box','value':'box'},
            ], value='scatter'),
        ], style={'width':'24%', 'display':'inline-block', 'marginLeft':'1%'}),
    ]),
    html.Div([
        html.Label("Filters (Pandas query syntax)"),
        dcc.Input(id='filter-query', type='text', placeholder="e.g., value > 10 and category == 'A'", style={'width':'60%'}),
        html.Button("Apply", id='apply-filter'),
        html.Button("Clear", id='clear-filter', style={'marginLeft':'6px'})
    ], style={'marginTop':12}),
    html.Div([
        html.Label("Group by (comma-separated columns)"),
        dcc.Input(id='group-by', type='text', placeholder="e.g., category", style={'width':'40%'}),
        html.Label("Aggregation (col:agg,...)"),
        dcc.Input(id='agg-expr', type='text', placeholder="e.g., value:mean,value:sum", style={'width':'40%'}),
        html.Button("Apply Group/Agg", id='apply-group', style={'display':'block', 'marginTop':'6px'})
    ], style={'marginTop':12}),
    html.Hr(),
    dcc.Loading(dcc.Graph(id='main-graph'), type='default'),
    html.Div([
        html.Button("Download plot as HTML", id='download-html'),
        dcc.Download(id='download-file')
    ], style={'marginTop':10}),
    # Hidden store for dataframe
    dcc.Store(id='df-store'),
    dcc.Store(id='filtered-df-store')
], style={'width':'95%', 'margin':'auto'})

# Callbacks
@app.callback(
    Output('df-store', 'data'),
    Output('file-info', 'children'),
    Input('upload-data', 'contents'),
    Input('load-sample', 'n_clicks'),
    State('upload-data', 'filename'),
    prevent_initial_call=False
)
def handle_load(contents, sample_clicks, filename):
    trigger = ctx.triggered_id
    try:
        if trigger == 'upload-data' and contents:
            content_type, content_string = contents.split(',',1)
            decoded = base64.b64decode(content_string)
            bio = io.BytesIO(decoded)
            # infer by filename
            name = filename or 'uploaded.csv'
            p = Path(name)
            if p.suffix.lower() in ('.csv', '.txt'):
                df = pd.read_csv(bio)
            elif p.suffix.lower() in ('.xls', '.xlsx'):
                df = pd.read_excel(bio)
            elif p.suffix.lower() == '.json':
                df = pd.read_json(bio)
            else:
                return dash.no_update, html.Div("Unsupported file type")
            info = html.Div(f"Loaded {name} — {df.shape[0]} rows × {df.shape[1]} cols")
        else:
            df = sample_df()
            info = html.Div(f"Sample dataset — {df.shape[0]} rows × {df.shape[1]} cols")
        # convert to json for storage
        return df.to_json(date_format='iso', orient='split'), info
    except Exception as e:
        return dash.no_update, html.Div(f"Error loading file: {e}")

# Populate column dropdowns when df changes
@app.callback(
    Output('x-col','options'),
    Output('y-col','options'),
    Output('color-col','options'),
    Input('df-store','data')
)
def populate_columns(df_json):
    if not df_json:
        return [], [], []
    df = pd.read_json(df_json, orient='split')
    opts = [{'label': c, 'value': c} for c in df.columns]
    return opts, opts, [{'label':'None','value':''}] + opts

# Apply filter
@app.callback(
    Output('filtered-df-store','data'),
    Input('apply-filter','n_clicks'),
    Input('clear-filter','n_clicks'),
    State('filter-query','value'),
    State('df-store','data'),
    prevent_initial_call=True
)
def apply_filter(apply_n, clear_n, query, df_json):
    trigger = ctx.triggered_id
    if not df_json:
        return dash.no_update
    df = pd.read_json(df_json, orient='split')
    if trigger == 'clear-filter':
        return df.to_json(orient='split')
    try:
        if query and query.strip():
            filtered = df.query(query)
        else:
            filtered = df
        return filtered.to_json(orient='split')
    except Exception:
        return df.to_json(orient='split')

# Update graph
@app.callback(
    Output('main-graph','figure'),
    Input('plot-type','value'),
    Input('x-col','value'),
    Input('y-col','value'),
    Input('color-col','value'),
    Input('filtered-df-store','data'),
    Input('apply-group','n_clicks'),
    State('group-by','value'),
    State('agg-expr','value'),
    prevent_initial_call=False
)
def update_graph(plot_type, xcol, ycol, color, filtered_json, group_clicks, group_by, agg_expr):
    if not filtered_json:
        return px.scatter()  # empty
    df = pd.read_json(filtered_json, orient='split')
    # Group/Agg if requested (only on apply click)
    if ctx.triggered_id == 'apply-group' and group_by and agg_expr:
        try:
            group_cols = [c.strip() for c in group_by.split(',') if c.strip()]
            agg_pairs = [p.strip() for p in agg_expr.split(',') if p.strip()]
            agg_dict = {}
            for p in agg_pairs:
                col, agg = [x.strip() for x in p.split(':')]
                agg_dict.setdefault(col, []).append(agg)
            df = df.groupby(group_cols).agg(agg_dict).reset_index()
        except Exception:
            pass

    # Basic plot choices
    try:
        if plot_type == 'scatter':
            fig = px.scatter(df, x=xcol, y=ycol, color=color or None, title=f"{ycol} vs {xcol}")
        elif plot_type == 'line':
            fig = px.line(df, x=xcol, y=ycol, color=color or None, title=f"{ycol} vs {xcol}")
        elif plot_type == 'hist':
            fig = px.histogram(df, x=xcol or ycol, color=color or None, nbins=40, title=f"Histogram of {xcol or ycol}")
        elif plot_type == 'bar':
            fig = px.bar(df, x=xcol, y=ycol, color=color or None, title=f"Bar: {ycol} by {xcol}")
        elif plot_type == 'box':
            fig = px.box(df, x=color or xcol, y=ycol or xcol, title="Box plot")
        else:
            fig = px.scatter(df, x=xcol, y=ycol, color=color or None)
        fig.update_layout(transition_duration=200)
        return fig
    except Exception as e:
        return px.scatter(title=f"Error plotting: {e}")

# Download plot as HTML
@app.callback(
    Output('download-file','data'),
    Input('download-html','n_clicks'),
    State('main-graph','figure'),
    prevent_initial_call=True
)
def download_html(n, fig):
    if not fig:
        return dash.no_update
    import json, tempfile
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.html')
    from plotly.offline import plot
    plot(fig, filename=tmp.name, auto_open=False)
    tmp.close()
    return dcc.send_file(tmp.name)

if __name__ == '__main__':
    # optional port argument
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 8050
    app.run_server(debug=True, port=port)